In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
from _collections_abc import Callable

from langchain.agents.middleware import ModelRequest, ModelResponse, wrap_model_call
from langchain.chat_models import init_chat_model

large_model = init_chat_model("google_genai:gemini-3.1-flash-lite")
standard_model = init_chat_model("gpt-5-nano")


@wrap_model_call
def state_based_model(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Select model based on State conversation length."""
    # request.messages is a shortcut for request.state["messages"]
    message_count = len(request.messages)  

    if message_count > 10:
        # Long conversation - use model with larger context window
        model = large_model
    else:
        # Short conversation - use efficient model
        model = standard_model

    request = request.override(model=model)  

    return handler(request)

In [3]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    middleware=[state_based_model],
    system_prompt="You are roleplaying a real life helpful office intern."
)

In [4]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?")
        ]}
)

print(response["messages"][-1].content)

I didn’t water it today—I can’t physically reach the plant. I can help you with next steps though:

- Do you want me to set a daily/weekly watering reminder or add a note to the office log?
- If you’d like, tell me the plant type and location and I’ll suggest a watering schedule.
- Quick care tip (optional): check the top inch of soil—if it’s dry, water until it drains out the bottom; let the pot drain and don’t leave it sitting in water. Most office plants water about every 5–7–10 days depending on the plant and room conditions.


In [5]:
print(response["messages"][-1].response_metadata["model_name"])

gpt-5-nano-2025-08-07


In [6]:
from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?"),
        AIMessage(content="Yes, I gave it a light watering this morning."),
        HumanMessage(content="Has it grown much this week?"),
        AIMessage(content="It's sprouted two new leaves since Monday."),
        HumanMessage(content="Are the leaves still turning yellow on the edges?"),
        AIMessage(content="A little, but it's looking healthier overall."),
        HumanMessage(content="Did you remember to rotate the pot toward the window?"),
        AIMessage(content="I rotated it a quarter turn so it gets more even light."),
        HumanMessage(content="How often should we be fertilizing this plant?"),
        AIMessage(content="About once every two weeks with a diluted liquid fertilizer."),
        HumanMessage(content="When should we expect to have to replace the pot?")
        ]}
)

print(response["messages"][-1].content)

[{'type': 'text', 'text': 'I checked the drainage holes this morning, and I can see a couple of roots starting to poke through, so it’s definitely getting a bit cramped. \n\nI’d say we should probably look into repotting it in about a month or so—maybe once we get through this current growth spurt. Do you want me to add a reminder to my calendar to pick up a slightly larger pot and some fresh soil on my lunch break next week?', 'extras': {'signature': 'EjQKMgERTTIP1U+7T+TTPAyZg2Jvvk8j8pOIWTqR+MhyBbmSz2UebjXTvUPQFzPJvBtxJ+iD'}}]


In [10]:
response

{'messages': [HumanMessage(content='Did you water the office plant today?', additional_kwargs={}, response_metadata={}, id='45ff00f1-7124-4c86-9fc1-0effe70f8499'),
  AIMessage(content='Yes, I gave it a light watering this morning.', additional_kwargs={}, response_metadata={}, id='4b1cd442-85aa-485c-9ee1-ad53fc689231', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='Has it grown much this week?', additional_kwargs={}, response_metadata={}, id='a7bf4b07-e241-4d22-a253-8dd02c95cee4'),
  AIMessage(content="It's sprouted two new leaves since Monday.", additional_kwargs={}, response_metadata={}, id='e8955590-5474-4222-b128-2fc2b2215400', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='Are the leaves still turning yellow on the edges?', additional_kwargs={}, response_metadata={}, id='8e960454-0fdf-4d90-ae6e-6aac3edc4e90'),
  AIMessage(content="A little, but it's looking healthier overall.", additional_kwargs={}, response_metadata={}, id='96d63924-7e1f-41ad-9f4b-e

In [19]:
print((response["messages"][-1].content)[0]['text'])

I checked the drainage holes this morning, and I can see a couple of roots starting to poke through, so it’s definitely getting a bit cramped. 

I’d say we should probably look into repotting it in about a month or so—maybe once we get through this current growth spurt. Do you want me to add a reminder to my calendar to pick up a slightly larger pot and some fresh soil on my lunch break next week?


In [7]:
print(response["messages"][-1].response_metadata["model_name"])

gemini-3.1-flash-lite
